<a href="https://colab.research.google.com/github/Ashwini9713/Digital-forensics-/blob/main/exp_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:


import re
import email
import difflib
from email import policy
from urllib.parse import urlparse
from dataclasses import dataclass, field
from datetime import datetime


# ---------------------------------------------------------------------
# 1. Reference data — trusted domains / known brand names for spoof checks
# ---------------------------------------------------------------------

TRUSTED_BRANDS = {
    "paypal": "paypal.com",
    "microsoft": "microsoft.com",
    "google": "google.com",
    "apple": "apple.com",
    "amazon": "amazon.com",
    "netflix": "netflix.com",
    "bankofamerica": "bankofamerica.com",
    "chase": "chase.com",
    "linkedin": "linkedin.com",
}

SUSPICIOUS_TLDS = {".zip", ".xyz", ".top", ".gq", ".tk", ".ml", ".cf", ".work", ".click", ".rest"}

URL_SHORTENERS = {"bit.ly", "tinyurl.com", "t.co", "goo.gl", "ow.ly", "is.gd", "buff.ly"}

URGENCY_KEYWORDS = [
    "urgent", "verify your account", "suspended", "immediately", "act now",
    "confirm your identity", "unusual activity", "click here", "limited time",
    "your account will be closed", "security alert", "password expires",
]


# ---------------------------------------------------------------------
# 2. Data model
# ---------------------------------------------------------------------

@dataclass
class URLFinding:
    url: str
    domain: str
    verdict: str          # "suspicious" | "legitimate" | "unknown"
    reasons: list = field(default_factory=list)


@dataclass
class EmailReport:
    filename: str
    subject: str
    from_display: str
    from_address: str
    return_path: str
    reply_to: str
    received_chain: list
    spf_result: str
    dkim_result: str
    dmarc_result: str
    spoof_flags: list = field(default_factory=list)
    url_findings: list = field(default_factory=list)
    urgency_hits: list = field(default_factory=list)
    risk_score: int = 0
    risk_label: str = "Low"


# ---------------------------------------------------------------------
# 3. Header parsing / sender verification
# ---------------------------------------------------------------------

def parse_headers(msg) -> dict:
    """Extract key headers, including authentication results."""
    auth_results = msg.get("Authentication-Results", "") or ""

    def extract_auth(mech):
        m = re.search(rf"{mech}=(\w+)", auth_results, re.IGNORECASE)
        return m.group(1).lower() if m else "none"

    received_chain = msg.get_all("Received", []) or []

    return {
        "subject": msg.get("Subject", ""),
        "from_raw": msg.get("From", ""),
        "return_path": msg.get("Return-Path", "").strip("<>"),
        "reply_to": msg.get("Reply-To", ""),
        "received_chain": received_chain,
        "spf": extract_auth("spf"),
        "dkim": extract_auth("dkim"),
        "dmarc": extract_auth("dmarc"),
    }


def split_display_and_address(from_raw: str):
    """'Display Name <user@domain.com>' -> ('Display Name', 'user@domain.com')"""
    m = re.match(r'^"?([^"<]*)"?\s*<([^>]+)>$', from_raw.strip())
    if m:
        return m.group(1).strip(), m.group(2).strip().lower()
    return "", from_raw.strip().lower()


def domain_of(addr: str) -> str:
    return addr.split("@")[-1].lower() if "@" in addr else addr.lower()


def levenshtein_ratio(a: str, b: str) -> float:
    return difflib.SequenceMatcher(None, a, b).ratio()


def detect_spoofing(display_name: str, from_addr: str, return_path: str, reply_to: str) -> list:
    """Look for mismatches that indicate a spoofed / impersonated sender."""
    flags = []
    from_domain = domain_of(from_addr)

    # a) Display name claims a brand, but the domain doesn't match it
    name_lower = display_name.lower()
    for brand, real_domain in TRUSTED_BRANDS.items():
        if brand in name_lower and real_domain not in from_domain:
            # allow close-but-not-exact lookalikes to also be flagged below
            flags.append(
                f"Display name mentions '{brand}' but From domain is "
                f"'{from_domain}' (expected something under '{real_domain}')"
            )

    # b) Lookalike / typosquatted domain (homoglyphs, extra hyphens, etc.)
    for brand, real_domain in TRUSTED_BRANDS.items():
        base = real_domain.split(".")[0]
        if base in from_domain and from_domain != real_domain:
            flags.append(f"Lookalike domain detected: '{from_domain}' resembles '{real_domain}'")
        elif levenshtein_ratio(from_domain, real_domain) > 0.75 and from_domain != real_domain:
            flags.append(f"From domain '{from_domain}' is a close typosquat of '{real_domain}'")

    # c) Return-Path / envelope sender differs from the visible From domain
    if return_path:
        rp_domain = domain_of(return_path)
        if rp_domain and rp_domain != from_domain:
            flags.append(f"Return-Path domain '{rp_domain}' differs from From domain '{from_domain}'")

    # d) Reply-To silently redirects replies to a different domain
    if reply_to:
        reply_addr = re.search(r"[\w.+-]+@[\w.-]+", reply_to)
        if reply_addr:
            reply_domain = domain_of(reply_addr.group())
            if reply_domain != from_domain:
                flags.append(f"Reply-To domain '{reply_domain}' differs from From domain '{from_domain}'")

    return flags


# ---------------------------------------------------------------------
# 4. URL extraction & classification
# ---------------------------------------------------------------------

URL_RE = re.compile(r'https?://[^\s"\'<>\)\]]+', re.IGNORECASE)
IP_HOST_RE = re.compile(r'^\d{1,3}(\.\d{1,3}){3}$')


def extract_urls(body: str) -> list:
    return list(dict.fromkeys(URL_RE.findall(body)))  # dedupe, preserve order


def classify_url(url: str) -> URLFinding:
    parsed = urlparse(url)
    host = parsed.hostname or ""
    reasons = []

    if IP_HOST_RE.match(host):
        reasons.append("Uses a raw IP address instead of a domain name")

    if host in URL_SHORTENERS:
        reasons.append(f"Uses URL shortener '{host}' which hides the real destination")

    for tld in SUSPICIOUS_TLDS:
        if host.endswith(tld):
            reasons.append(f"Uses commonly-abused TLD '{tld}'")

    if host.count("-") >= 3:
        reasons.append("Domain contains an unusually high number of hyphens")

    if len(host.split(".")) > 4:
        reasons.append("Domain has an unusually deep subdomain structure")

    for brand, real_domain in TRUSTED_BRANDS.items():
        if brand in host and host != real_domain and not host.endswith("." + real_domain):
            reasons.append(f"Mentions brand '{brand}' but is not the real '{real_domain}' domain")

    if "@" in url:
        reasons.append("Contains '@' before the actual domain (classic redirection trick)")

    if re.search(r"\d{5,}", host):
        reasons.append("Domain contains an unusually long numeric sequence")

    verdict = "suspicious" if reasons else "unknown"
    return URLFinding(url=url, domain=host, verdict=verdict, reasons=reasons)


# ---------------------------------------------------------------------
# 5. Urgency / social-engineering language detection
# ---------------------------------------------------------------------

def detect_urgency_language(text: str) -> list:
    text_lower = text.lower()
    return [kw for kw in URGENCY_KEYWORDS if kw in text_lower]


# ---------------------------------------------------------------------
# 6. Risk scoring
# ---------------------------------------------------------------------

def score_report(spoof_flags, url_findings, urgency_hits, spf, dkim, dmarc) -> tuple:
    score = 0
    score += 15 * len(spoof_flags)
    score += sum(8 + 3 * len(f.reasons) for f in url_findings if f.verdict == "suspicious")
    score += 5 * len(urgency_hits)
    for result in (spf, dkim, dmarc):
        if result in ("fail", "none"):
            score += 10

    score = min(score, 100)
    if score >= 60:
        label = "High"
    elif score >= 30:
        label = "Medium"
    else:
        label = "Low"
    return score, label


# ---------------------------------------------------------------------
# 7. Main analysis pipeline
# ---------------------------------------------------------------------

def analyze_email(raw_text: str, filename: str = "sample.eml") -> EmailReport:
    msg = email.message_from_string(raw_text, policy=policy.default)
    headers = parse_headers(msg)
    display_name, from_addr = split_display_and_address(headers["from_raw"])

    # Get plain-text body
    if msg.is_multipart():
        body = ""
        for part in msg.walk():
            if part.get_content_type() == "text/plain":
                body += part.get_content()
    else:
        body = msg.get_content()

    spoof_flags = detect_spoofing(display_name, from_addr, headers["return_path"], headers["reply_to"])
    urls = extract_urls(body)
    url_findings = [classify_url(u) for u in urls]
    urgency_hits = detect_urgency_language(headers["subject"] + " " + body)

    score, label = score_report(
        spoof_flags, url_findings, urgency_hits,
        headers["spf"], headers["dkim"], headers["dmarc"]
    )

    return EmailReport(
        filename=filename,
        subject=headers["subject"],
        from_display=display_name,
        from_address=from_addr,
        return_path=headers["return_path"],
        reply_to=headers["reply_to"],
        received_chain=headers["received_chain"],
        spf_result=headers["spf"],
        dkim_result=headers["dkim"],
        dmarc_result=headers["dmarc"],
        spoof_flags=spoof_flags,
        url_findings=url_findings,
        urgency_hits=urgency_hits,
        risk_score=score,
        risk_label=label,
    )


# ---------------------------------------------------------------------
# 8. Reporting
# ---------------------------------------------------------------------

def print_report(r: EmailReport):
    print("=" * 70)
    print(f"FILE: {r.filename}")
    print(f"Subject: {r.subject}")
    print(f"From: \"{r.from_display}\" <{r.from_address}>")
    print(f"Return-Path: {r.return_path or '(none)'}")
    print(f"Reply-To: {r.reply_to or '(none)'}")
    print(f"SPF={r.spf_result}  DKIM={r.dkim_result}  DMARC={r.dmarc_result}")
    print("-" * 70)

    print(f"Sender spoofing flags ({len(r.spoof_flags)}):")
    for f in r.spoof_flags:
        print(f"  - {f}")
    if not r.spoof_flags:
        print("  (none detected)")

    print(f"\nURLs found ({len(r.url_findings)}):")
    for f in r.url_findings:
        print(f"  [{f.verdict.upper()}] {f.url}")
        for reason in f.reasons:
            print(f"      -> {reason}")

    print(f"\nUrgency / social-engineering phrases: {r.urgency_hits or '(none)'}")
    print(f"\nRISK SCORE: {r.risk_score}/100  ->  {r.risk_label.upper()} RISK")
    print("=" * 70 + "\n")


def summary_statistics(reports: list):
    total = len(reports)
    high = sum(1 for r in reports if r.risk_label == "High")
    med = sum(1 for r in reports if r.risk_label == "Medium")
    low = sum(1 for r in reports if r.risk_label == "Low")
    total_urls = sum(len(r.url_findings) for r in reports)
    suspicious_urls = sum(1 for r in reports for f in r.url_findings if f.verdict == "suspicious")
    spoofed = sum(1 for r in reports if r.spoof_flags)
    auth_fail = sum(1 for r in reports if "fail" in (r.spf_result, r.dkim_result, r.dmarc_result))

    print("#" * 70)
    print("SUMMARY STATISTICS")
    print("#" * 70)
    print(f"Emails analyzed:            {total}")
    print(f"  High risk:                {high}")
    print(f"  Medium risk:              {med}")
    print(f"  Low risk:                 {low}")
    print(f"Emails with spoofed sender: {spoofed}")
    print(f"Emails failing SPF/DKIM/DMARC: {auth_fail}")
    print(f"Total URLs extracted:       {total_urls}")
    print(f"Suspicious URLs flagged:    {suspicious_urls}")
    print("#" * 70)


# ---------------------------------------------------------------------
# 9. Sample phishing/legit emails for demonstration
#    (synthetic, for classroom use only)
# ---------------------------------------------------------------------

SAMPLE_1 = """From: "PayPal Security" <security@paypa1-support.com>
To: victim@example.com
Subject: Urgent: Your account will be suspended - Verify your account immediately
Return-Path: <bounce@totally-different-domain.ru>
Reply-To: scammer@paypa1-support.com
Authentication-Results: mx.example.com; spf=fail; dkim=fail; dmarc=fail
Received: from unknown (unknown [185.220.101.50]) by mx.example.com

Dear Customer,

We have detected unusual activity on your account. Your account will be
closed if you do not verify your identity immediately. Click here to
confirm your identity:

http://paypa1-support-verify.xyz/login?redirect=http://192.168.55.10/steal

Act now, this is time sensitive.

PayPal Security Team
"""

SAMPLE_2 = """From: "LinkedIn" <messages-noreply@linkedin.com>
To: user@example.com
Subject: You have 3 new connection requests
Return-Path: <bounce@linkedin.com>
Authentication-Results: mx.example.com; spf=pass; dkim=pass; dmarc=pass
Received: from mail.linkedin.com (mail.linkedin.com [108.174.10.10]) by mx.example.com

Hi there,

You have new connection requests waiting. View them here:
https://www.linkedin.com/comm/mynetwork

Thanks,
The LinkedIn Team
"""

SAMPLE_3 = """From: "Microsoft Support" <support@micros0ft-alerts.top>
To: victim@example.com
Subject: Security Alert: unusual sign-in activity detected
Return-Path: <bounce@micros0ft-alerts.top>
Reply-To: help@micr0-billing-center.gq
Authentication-Results: mx.example.com; spf=none; dkim=none; dmarc=none
Received: from 10-9-8-7.static.example (unknown [45.33.12.9]) by mx.example.com

We noticed unusual activity on your Microsoft account. Please confirm your
identity within 24 hours or your account will be suspended.

Verify now: http://bit.ly/ms-verify-account

Regards,
Microsoft Account Team
"""


if __name__ == "__main__":
    samples = {
        "sample1_paypal_phish.eml": SAMPLE_1,
        "sample2_linkedin_legit.eml": SAMPLE_2,
        "sample3_microsoft_phish.eml": SAMPLE_3,
    }

    reports = []
    for fname, raw in samples.items():
        report = analyze_email(raw, filename=fname)
        reports.append(report)
        print_report(report)

    summary_statistics(reports)

FILE: sample1_paypal_phish.eml
Subject: Urgent: Your account will be suspended - Verify your account immediately
From: "PayPal Security" <security@paypa1-support.com>
Return-Path: bounce@totally-different-domain.ru
Reply-To: scammer@paypa1-support.com
SPF=fail  DKIM=fail  DMARC=fail
----------------------------------------------------------------------
Sender spoofing flags (2):
  - Display name mentions 'paypal' but From domain is 'paypa1-support.com' (expected something under 'paypal.com')
  - Return-Path domain 'totally-different-domain.ru' differs from From domain 'paypa1-support.com'

URLs found (1):
  [SUSPICIOUS] http://paypa1-support-verify.xyz/login?redirect=http://192.168.55.10/steal
      -> Uses commonly-abused TLD '.xyz'

Urgency / social-engineering phrases: ['urgent', 'verify your account', 'suspended', 'immediately', 'act now', 'confirm your identity', 'unusual activity', 'click here']

RISK SCORE: 100/100  ->  HIGH RISK

FILE: sample2_linkedin_legit.eml
Subject: You ha